# 🧠 Neuronske Mreže - Kompletan Vodič
## Predikcija, Matrica Konfuzije i Vizuelizacija

---

## 📦 Korak 1: Učitaj sve potrebne biblioteke

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# → Metrike iz sklearn
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, auc, roc_auc_score
)

warnings.filterwarnings('ignore')

# → Podešavanje stilova
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print('✓ Sve biblioteke učitane!')

✓ Sve biblioteke učitane!


## 🎯 Korak 2: Predikcija Modela

### Osnovna funkcija za predikciju i evaluaciju

In [3]:
def predict_and_evaluate(model, X_test, y_test, model_name=''):
    """
    Pravi predikcije na test setu i vraća sve potrebne metrike.
    
    PARAMETRI:
      → model: Treniran MLPClassifier ili MLPRegressor
      → X_test: Test features
      → y_test: Test target
      → model_name: Naziv modela (za ispis)
    
    VRAĆA:
      → dict sa predikcijama i svim metrikama
    """
    
    # → Predikcija
    y_pred = model.predict(X_test)
    
    # → Za klasifikaciju, dobijamo i verovatnoće
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)
    else:
        y_proba = None
    
    # → Osnovne metrike za klasifikaciju
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    rezultat = {
        'y_pred': y_pred,
        'y_proba': y_proba,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'model_name': model_name
    }
    
    # → Ispis rezultata
    print(f'\n{"═" * 70}')
    print(f'REZULTATI MODELA: {model_name}')
    print(f'{"═" * 70}')
    print(f'  Accuracy:  {accuracy:.4f}')
    print(f'  Precision: {precision:.4f}')
    print(f'  Recall:    {recall:.4f}')
    print(f'  F1-Score:  {f1:.4f}')
    print(f'{"═" * 70}\n')
    
    return rezultat

print('✓ Funkcija predict_and_evaluate učitana!')

✓ Funkcija predict_and_evaluate učitana!


## 📊 Korak 3: Matrica Konfuzije

### Binarna klasifikacija (2 klase)

In [4]:
def plot_confusion_matrix(y_test, y_pred, model_name='', figsize=(8, 6)):
    """
    Pravi i prikazuje matricu konfuzije sa heatmap vizuelizacijom.
    
    PARAMETRI:
      → y_test: Stvarne vrednosti
      → y_pred: Predikovane vrednosti
      → model_name: Naziv modela (za naslov)
      → figsize: Veličina figure (width, height)
    """
    
    # → Izračunaj matricu konfuzije
    cm = confusion_matrix(y_test, y_pred)
    
    # → Kreiraj figuru
    plt.figure(figsize=figsize)
    
    # → Nacrtaj heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
                xticklabels=['Negativna', 'Pozitivna'],
                yticklabels=['Negativna', 'Pozitivna'],
                annot_kws={'size': 14, 'weight': 'bold'})
    
    plt.title(f'Matrica Konfuzije - {model_name}', fontsize=14, fontweight='bold', pad=20)
    plt.ylabel('Stvarna vrednost', fontsize=12)
    plt.xlabel('Predikovana vrednost', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # → Detaljne metrike iz matrice
    print(f'\n{"─" * 70}')
    print(f'ANALIZA MATRICE KONFUZIJE - {model_name}')
    print(f'{"─" * 70}')
    
    tn, fp, fn, tp = cm.ravel()
    
    print(f'\n  Tačno Negativne (TN):     {tn:5d}  → Korektno rekao "NEĆE se promeniti"')
    print(f'  Pogrešno Pozitivne (FP):  {fp:5d}  → Rekao "HOĆE" ali nije')
    print(f'  Pogrešno Negativne (FN):  {fn:5d}  → Rekao "NEĆE" ali hoće')
    print(f'  Tačno Pozitivne (TP):     {tp:5d}  → Korektno rekao "HOĆE se promeniti"')
    
    # → Dodatne metrike
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    print(f'\n  Specifičnost (TNR):    {specificity:.4f}  → Detektuje negativne')
    print(f'  Osetljivost (TPR):     {sensitivity:.4f}  → Detektuje pozitivne')
    print(f'{"─" * 70}\n')
    
    return cm

print('✓ Funkcija plot_confusion_matrix učitana!')

✓ Funkcija plot_confusion_matrix učitana!


### Višeklasna klasifikacija (3+ klase)

In [5]:
def plot_confusion_matrix_multiclass(y_test, y_pred, class_names=None, model_name='', figsize=(10, 8)):
    """
    Matrica konfuzije za višeklasnu klasifikaciju.
    
    PARAMETRI:
      → y_test: Stvarne vrednosti
      → y_pred: Predikovane vrednosti
      → class_names: Lista naziva klasa
      → model_name: Naziv modela
    """
    cm = confusion_matrix(y_test, y_pred)
    
    if class_names is None:
        class_names = [f'Class {i}' for i in range(len(cm))]
    
    plt.figure(figsize=figsize)
    sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn', cbar=True,
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={'size': 10, 'weight': 'bold'})
    
    plt.title(f'Matrica Konfuzije (Višeklasna) - {model_name}', 
              fontsize=14, fontweight='bold', pad=20)
    plt.ylabel('Stvarna vrednost', fontsize=12)
    plt.xlabel('Predikovana vrednost', fontsize=12)
    plt.tight_layout()
    plt.show()

print('✓ Funkcija plot_confusion_matrix_multiclass učitana!')

✓ Funkcija plot_confusion_matrix_multiclass učitana!


## 🧠 Korak 4: Vizuelizacija Neuronske Mreže

### Dijagram arhitekture sa slojevima i neuronima

In [6]:
def visualize_network_architecture(model, input_dim, output_dim, figsize=(12, 8)):
    """
    Prikazuje arhitekturu neuronske mreže kao dijagram.
    
    PARAMETRI:
      → model: Treniran MLPClassifier ili MLPRegressor
      → input_dim: Broj input karakteristika
      → output_dim: Broj output klasa/neurona
    """
    
    # → Izvuci informacije o arhitekturi
    layers = [input_dim] + list(model.hidden_layer_sizes) + [output_dim]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # → Parametri crtanja
    layer_width = 1.0
    neuron_radius = 0.3
    
    # → Crtaj slojeve
    layer_positions = {}
    x_positions = np.linspace(0, len(layers) - 1, len(layers))
    
    for layer_idx, (x, num_neurons) in enumerate(zip(x_positions, layers)):
        layer_positions[layer_idx] = []
        
        # → Y pozicije za neurone u sloju
        if num_neurons == 1:
            y_positions = [0]
        else:
            y_positions = np.linspace(-num_neurons/2, num_neurons/2, num_neurons)
        
        # → Crtaj neurone
        for y in y_positions:
            circle = plt.Circle((x, y), neuron_radius, color='#3498db', ec='#2c3e50', 
                               linewidth=2, zorder=3)
            ax.add_patch(circle)
            layer_positions[layer_idx].append((x, y))
    
    # → Crtaj veze između neurona
    for layer_idx in range(len(layers) - 1):
        for x1, y1 in layer_positions[layer_idx]:
            for x2, y2 in layer_positions[layer_idx + 1]:
                ax.plot([x1, x2], [y1, y2], 'gray', alpha=0.3, linewidth=1, zorder=1)
    
    # → Labele za slojeve
    layer_names = (['Input'] + 
                  [f'Hidden {i+1}' for i in range(len(model.hidden_layer_sizes))] + 
                  ['Output'])
    
    max_neurons = max([len(positions) for positions in layer_positions.values()])
    
    for x, name in zip(x_positions, layer_names):
        ax.text(x, -max_neurons/2 - 1.5, name, ha='center', fontsize=11, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.5', facecolor='lightgray', alpha=0.7))
    
    # → Labele za broj neurona
    for layer_idx, (x, num_neurons) in enumerate(zip(x_positions, layers)):
        ax.text(x, max_neurons/2 + 1, f'{num_neurons}', ha='center', fontsize=10, 
               color='white', fontweight='bold',
               bbox=dict(boxstyle='circle,pad=0.3', facecolor='#e74c3c', alpha=0.8))
    
    # → Postavi parametre grafa
    ax.set_xlim(-1, len(layers))
    ax.set_ylim(-max_neurons/2 - 2, max_neurons/2 + 2)
    ax.set_aspect('equal')
    ax.axis('off')
    
    # → Naslov sa detaljima
    activation = getattr(model, 'activation', 'unknown')
    solver = getattr(model, 'solver', 'unknown')
    
    plt.title(f'Arhitektura Neuronske Mreže\n' + 
             f'Aktivacija: {activation} | Solver: {solver} | ' +
             f'Iteracija: {model.n_iter_}',
             fontsize=13, fontweight='bold', pad=20)
    
    plt.tight_layout()
    plt.show()

print('✓ Funkcija visualize_network_architecture učitana!')

✓ Funkcija visualize_network_architecture učitana!


## 📈 Korak 5: ROC Kriva

### Receiver Operating Characteristic kriva i AUC skor

In [7]:
def plot_roc_curve(y_test, y_proba, model_name='', figsize=(8, 6)):
    """
    Pravi ROC krivu (Receiver Operating Characteristic).
    
    PARAMETRI:
      → y_test: Stvarne vrednosti
      → y_proba: Verovatnoće predviđanja (iz predict_proba)
      → model_name: Naziv modela
    """
    
    if y_proba is None or len(y_proba.shape) == 1:
        print('⚠️  ROC kriva zahteva verovatnoće iz predict_proba()')
        return
    
    # → Izvuci verovatnoće za pozitivnu klasu
    if y_proba.shape[1] > 1:
        y_proba_pos = y_proba[:, 1]
    else:
        return
    
    # → Izračunaj ROC krivu
    fpr, tpr, thresholds = roc_curve(y_test, y_proba_pos)
    roc_auc = auc(fpr, tpr)
    
    # → Crtaj
    plt.figure(figsize=figsize)
    plt.plot(fpr, tpr, color='#3498db', lw=2.5, 
            label=f'ROC Kriva (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', 
            label='Slučajni klasifikator')
    
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Lažno Pozitivna Stopa (FPR)', fontsize=11)
    plt.ylabel('Tačno Pozitivna Stopa (TPR)', fontsize=11)
    plt.title(f'ROC Kriva - {model_name}', fontsize=13, fontweight='bold', pad=20)
    plt.legend(loc='lower right', fontsize=11)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f'\n🎯 AUC Skor: {roc_auc:.4f}')
    print(f'   → 0.5 = nasumičan klasifikator')
    print(f'   → 1.0 = savršena klasifikacija')

print('✓ Funkcija plot_roc_curve učitana!')

✓ Funkcija plot_roc_curve učitana!


## 🏆 Korak 6: Poređenje Više Modela

### Bar chart i heatmap poređenja performansi

In [9]:
def compare_models(models_dict, X_test, y_test, figsize=(14, 5)):
    """
    Poređenje više modela na istom test setu.
    
    PARAMETRI:
      → models_dict: Dict sa {'naziv_modela': model}
      → X_test: Test features
      → y_test: Test target
    """
    
    results = {}
    
    # → Iteracija kroz sve modele
    for name, model in models_dict.items():
        y_pred = model.predict(X_test)
        
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        
        results[name] = {
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1
        }
    
    # → Kreiraj DataFrame
    df_results = pd.DataFrame(results).T
    
    # → Crtaj
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # → Bar chart
    df_results.plot(kind='bar', ax=axes[0], 
                   color=['#3498db', '#e74c3c', '#2ecc71', '#f39c12'])
    axes[0].set_title('Poređenje Metrika Modela', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Skor', fontsize=11)
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
    axes[0].grid(alpha=0.3, axis='y')
    axes[0].legend(loc='lower right')
    axes[0].set_ylim([0, 1.1])
    
    # → Heatmap
    sns.heatmap(df_results, annot=True, fmt='.4f', cmap='RdYlGn', 
               ax=axes[1], cbar_kws={'label': 'Skor'}, vmin=0, vmax=1)
    axes[1].set_title('Heatmap Metrika', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # → Ispis tabele
    print('\n' + '═' * 80)
    print('POREĐENJE MODELA')
    print('═' * 80)
    print(df_results.to_string())
    print('═' * 80 + '\n')
    
    # → Pronađi najbolji model
    best_model = df_results['F1-Score'].idxmax()
    best_score = df_results['F1-Score'].max()
    print(f'🏆 Najbolji model: {best_model} (F1-Score: {best_score:.4f})')
    
    return df_results

print('✓ Funkcija compare_models učitana!')

✓ Funkcija compare_models učitana!


## 🔍 Korak 7: Analiza Greške

### Pronađi gde model pravi greške

In [10]:
def error_analysis(y_test, y_pred, X_test, feature_names=None):
    """
    Analizira gde model pravi greške.
    
    PARAMETRI:
      → y_test: Stvarne vrednosti
      → y_pred: Predikovane vrednosti
      → X_test: Test features
      → feature_names: Nazivi features (opcionalno)
    """
    
    # → Pronađi gde je model pogrešio
    errors = y_test != y_pred
    error_indices = np.where(errors)[0]
    correct_indices = np.where(~errors)[0]
    
    print(f'\n{"─" * 70}')
    print(f'ANALIZA GREŠKE')
    print(f'{"─" * 70}')
    
    total = len(y_test)
    error_count = len(error_indices)
    error_pct = error_count / total * 100 if total > 0 else 0
    
    print(f'\n📌 Ukupno grešaka: {error_count} od {total} ({error_pct:.2f}%)')
    
    if error_count == 0:
        print('✓ Nema grešaka! Model je savršen!')
        return
    
    # → Poređenje prosečnih vrednosti
    print(f'\n📊 Prosečne vrednosti za GREŠKE:')
    error_means = X_test.iloc[error_indices].mean() if isinstance(X_test, pd.DataFrame) else X_test[error_indices].mean(axis=0)
    
    print(f'\n📊 Prosečne vrednosti za TAČNE predikcije:')
    correct_means = X_test.iloc[correct_indices].mean() if isinstance(X_test, pd.DataFrame) else X_test[correct_indices].mean(axis=0)
    
    # → Pronađi maximum razlike
    if isinstance(X_test, pd.DataFrame):
        diff = (error_means - correct_means).abs().sort_values(ascending=False)
        print(f'\n🔴 Top 5 features sa najvećom razlikom između grešaka i tačnih:')
        for i, (feat, val) in enumerate(diff.head(5).items(), 1):
            print(f'   {i}. {feat}: {val:.4f}')
    
    print(f'{"─" * 70}\n')

print('✓ Funkcija error_analysis učitana!')

✓ Funkcija error_analysis učitana!


## 📋 Korak 8: Detaljna Analiza Metrika

### Detaljni classification report

In [11]:
def detailed_metrics_report(y_test, y_pred, target_names=None, model_name=''):
    """
    Prikazuje detaljnu analizu svih metrika.
    
    PARAMETRI:
      → y_test: Stvarne vrednosti
      → y_pred: Predikovane vrednosti
      → target_names: Nazivi klasa
      → model_name: Naziv modela
    """
    
    if target_names is None:
        target_names = [f'Class {i}' for i in range(len(np.unique(y_test)))]
    
    print(f'\n{"═" * 80}')
    print(f'DETALJNE METRIKE - {model_name}')
    print(f'{"═" * 80}\n')
    
    # → Classification report
    report = classification_report(y_test, y_pred, target_names=target_names)
    print(report)
    
    # → DataFrame verzija za lakše čitanje
    from sklearn.metrics import precision_recall_fscore_support
    
    precision, recall, f1, support = precision_recall_fscore_support(
        y_test, y_pred, average=None
    )
    
    df_metrics = pd.DataFrame({
        'Klasa': target_names,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Podrška': support
    })
    
    print('\n📊 TABELA METRIKA:\n')
    print(df_metrics.to_string(index=False))
    print(f'\n{"═" * 80}\n')
    
    return df_metrics

print('✓ Funkcija detailed_metrics_report učitana!')

✓ Funkcija detailed_metrics_report učitana!


---

## 🚀 PRIMER KORIŠĆENJA

### Preuzmi primer kako koristiti sve funkcije

**KORAK 1:** Trenira model

```python
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation='relu',
    solver='adam',
    max_iter=500,
    early_stopping=True
)
model.fit(X_train, y_train)
```

**KORAK 2:** Primeni sve funkcije odjednom

In [12]:
# ═══════════════════════════════════════════════════════════════════════════════
# KOMPLETAN PRIMER KORIŠĆENJA SVIH FUNKCIJA
# ═══════════════════════════════════════════════════════════════════════════════

# Pretpostavljam da već imaš:
# - model_classifier (treniran MLP model)
# - X_test, y_test (test dataset)
# - X_train (za određivanje dimenzija)

# Ako NEMAŠ model, kreiraj test primer:

# from sklearn.datasets import make_classification
# from sklearn.model_selection import train_test_split
# from sklearn.neural_network import MLPClassifier
# 
# # Kreiraj test dataset
# X, y = make_classification(n_samples=200, n_features=20, n_informative=10, 
#                            n_redundant=5, n_classes=2, random_state=42)
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# 
# # Trenira model
# model_classifier = MLPClassifier(hidden_layer_sizes=(32, 16), 
#                                  activation='relu', solver='adam', max_iter=500)
# model_classifier.fit(X_train, y_train)
# 
# print('✓ Model trenirani!')

print('\n' + '═' * 80)
print('PRIMER KORIŠĆENJA - KOMENTISITE REDOVE ISPOD AKO IMATE SVOJ MODEL')
print('═' * 80 + '\n')

# # KORAK 1: PREDIKCIJA
# print('\n▶️ KORAK 1: PREDIKCIJA')
# rezultat = predict_and_evaluate(model_classifier, X_test, y_test, 'Moj MLP Model')

# # KORAK 2: MATRICA KONFUZIJE
# print('\n▶️ KORAK 2: MATRICA KONFUZIJE')
# plot_confusion_matrix(y_test, rezultat['y_pred'], 'Moj MLP Model')

# # KORAK 3: ARHITEKTURA MREŽE
# print('\n▶️ KORAK 3: ARHITEKTURA NEURONSKE MREŽE')
# visualize_network_architecture(model_classifier, X_train.shape[1], 2)

# # KORAK 4: ROC KRIVA
# print('\n▶️ KORAK 4: ROC KRIVA')
# plot_roc_curve(y_test, rezultat['y_proba'], 'Moj MLP Model')

# # KORAK 5: DETALJNE METRIKE
# print('\n▶️ KORAK 5: DETALJNE METRIKE')
# df_metrics = detailed_metrics_report(y_test, rezultat['y_pred'], 
#                                      target_names=['Negativna', 'Pozitivna'],
#                                      model_name='Moj MLP Model')

# # KORAK 6: ANALIZA GREŠKE
# print('\n▶️ KORAK 6: ANALIZA GREŠKE')
# error_analysis(y_test, rezultat['y_pred'], X_test)

# # KORAK 7: POREĐENJE MODELA (ako imaš više modela)
# print('\n▶️ KORAK 7: POREĐENJE MODELA')
# models = {
#     'Model 1': model_classifier,
#     # 'Model 2': model2,  # ako imaš drugi model
#     # 'Model 3': model3   # ako imaš treći model
# }
# df_comparison = compare_models(models, X_test, y_test)

print('✓ Sve funkcije su aktivne!')
print('\n💡 Predlog: Odkomentiši redove gore (ukloni #) i pokreni ovu ćeliju sa tvojim podacima!')


════════════════════════════════════════════════════════════════════════════════
PRIMER KORIŠĆENJA - KOMENTISITE REDOVE ISPOD AKO IMATE SVOJ MODEL
════════════════════════════════════════════════════════════════════════════════

✓ Sve funkcije su aktivne!

💡 Predlog: Odkomentiši redove gore (ukloni #) i pokreni ovu ćeliju sa tvojim podacima!


---

## 📚 CHEAT SHEET - Brza Referenca

| Funkcija | Namena | Parametri |
|----------|--------|----------|
| `predict_and_evaluate()` | Predikcija + sve metrike | `(model, X_test, y_test, model_name)` |
| `plot_confusion_matrix()` | Matrica konfuzije (2 klase) | `(y_test, y_pred, model_name)` |
| `plot_confusion_matrix_multiclass()` | Matrica konfuzije (3+ klase) | `(y_test, y_pred, class_names, model_name)` |
| `visualize_network_architecture()` | Dijagram mreže | `(model, input_dim, output_dim)` |
| `plot_roc_curve()` | ROC kriva i AUC | `(y_test, y_proba, model_name)` |
| `detailed_metrics_report()` | Detaljne metrike | `(y_test, y_pred, target_names, model_name)` |
| `error_analysis()` | Gde model pravi greške | `(y_test, y_pred, X_test)` |
| `compare_models()` | Poređenje više modela | `(models_dict, X_test, y_test)` |

---

## 🎯 Što znače metrike?

- **Accuracy** → Koliko je model tačan (0-1)
- **Precision** → Od pozitivnih predikcija, koliko je stvarno pozitivno
- **Recall** → Od stvarnih pozitivnih, koliko je model našao
- **F1-Score** → Kombinovana balansa između Precision i Recall
- **AUC** → Površina ispod ROC krive (bolje = bliže 1.0)

---

## 💡 Saveti

1. **Počni sa predikcijom** → `predict_and_evaluate()` da vidiš osnovne metrike
2. **Vizuelizuj problem** → `plot_confusion_matrix()` da vidiš gde model greši
3. **Analiza greške** → `error_analysis()` da shvatiš pattern grešaka
4. **Poredi modele** → `compare_models()` da vidiš koji je najbolji
5. **ROC kriva** → `plot_roc_curve()` da vidiš sveukupnu performansu